# RabTech Academy – Task 4
## Dimensionality Reduction & Unsupervised Clustering

**Dataset:** UCI Student Performance – Mathematics (`student-mat.csv`)

### Aim
Use Principal Component Analysis (PCA) to reduce the dimensionality of student data, then apply K-Means, DBSCAN, and Agglomerative (hierarchical) clustering to explore student profiles.

### Research questions
1. How many principal components are needed to explain most of the variation in the student features?
2. What number of clusters is a reasonable choice for K-Means according to the elbow curve and silhouette score?
3. What student groups are identified by K-Means, DBSCAN, and hierarchical clustering?
4. How do the discovered groups differ in final mathematics grade (`G3`)?

### Important analysis decisions
- `G3` (final grade) is kept out of the clustering input and used only after clustering to describe the groups. This avoids defining clusters using the outcome we later compare.
- The original dataset has mixed numeric and categorical columns. Categorical predictor columns are one-hot encoded, and all resulting features are standardized.
- PCA is fitted on the standardized feature matrix.
- K-Means is evaluated using inertia (elbow method) and silhouette score.
- DBSCAN is run on a PCA representation. Its `eps` value is a parameter to investigate, not a universally correct value.
- Hierarchical clustering is also run on the PCA representation.
- Clustering is exploratory. It does not prove that naturally distinct student types exist, and it does not establish cause and effect.

## 1. Get the dataset

Download `student-mat.csv` from the official UCI Student Performance dataset page:

https://archive.ics.uci.edu/dataset/320/student+performance

Create this folder structure beside the notebook:

```text
project-folder/
│
├── Task4_Dimensionality_Reduction_Clustering.ipynb
├── requirements-task4.txt
└── data/
    └── raw/
        └── student-mat.csv
```

The file is semicolon-separated, so the code below uses `sep=";"`.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
DATA_PATH = Path("data/raw/student-mat.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Download student-mat.csv from UCI and place it in data/raw/"
    )

raw = pd.read_csv(DATA_PATH, sep=";")
print("Dataset shape:", raw.shape)
print("Columns:", list(raw.columns))
display(raw.head())

## 2. Prepare the data

We remove exact duplicate rows and rows with missing values. `G3` is saved separately for later cluster interpretation and is not used to create the clusters.

All other columns are used as predictors. Categorical columns are converted to indicator columns using one-hot encoding. Standardization is important because features measured on different scales should not dominate PCA or distance-based clustering.

In [ ]:
df = raw.drop_duplicates().copy()

if "G3" not in df.columns:
    raise ValueError("The dataset must contain the G3 final-grade column.")

grade = pd.to_numeric(df["G3"], errors="coerce")
X_raw = df.drop(columns=["G3"]).copy()

# Convert categorical columns into one-hot numerical columns.
X = pd.get_dummies(X_raw, drop_first=True, dtype=float)

# Ensure numeric values and remove rows with missing/infinite values.
X = X.apply(pd.to_numeric, errors="coerce")
valid = X.notna().all(axis=1) & np.isfinite(X).all(axis=1) & grade.notna()

X = X.loc[valid].copy()
grade = grade.loc[valid].copy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Rows used:", X.shape[0])
print("Numerical features after encoding:", X.shape[1])
print("Rows excluded:", len(raw) - len(X))
print("Final grade is excluded from clustering features:", "G3" not in X.columns)

## 3. Principal Component Analysis (PCA)

PCA creates new, uncorrelated components from the standardized features. The explained variance ratio tells us how much information each component accounts for.

First, fit PCA without limiting the number of components. Then inspect the scree plot and cumulative explained variance to decide how many components to retain.

In [ ]:
pca_full = PCA()
X_pca_full = pca_full.fit_transform(X_scaled)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(explained) + 1), explained, marker="o,")
plt.title("PCA Scree Plot")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance Ratio")
plt.grid(True)
plt.show()

plt.figure(figsize=(9, 5))
plt.plot(range(1, len(cumulative) + 1), cumulative, marker="o")
plt.axhline(0.90, linestyle="--", label="90% variance")
plt.title("Cumulative Explained Variance")
plt.xlabel("Number of Principal Components")
plt.ylabel("Cumulative Explained Variance")
plt.legend()
plt.grid(True)
plt.show()

n_90 = int(np.argmax(cumulative >= 0.90) + 1)
print("Components needed to explain at least 90% variance:", n_90)
print("Variance explained by first 2 components:", round(cumulative[min(1, len(cumulative)-1)], 4))

## 4. Retain PCA components

For clustering and visualization, retain enough components to explain at least 90% of the variance. If that requires many components, the notebook still keeps them for clustering; the first two or three are used only for plots.

In [ ]:
pca = PCA(n_components=0.90, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)

print("Original feature count:", X_scaled.shape[1])
print("Retained component count:", X_pca.shape[1])
print("Total explained variance:", round(pca.explained_variance_ratio_.sum(), 4))

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d = pca_2d.fit_transform(X_scaled)

if X_scaled.shape[1] >= 3:
    pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
    X_3d = pca_3d.fit_transform(X_scaled)

## 5. K-Means: Elbow method and Silhouette analysis

K-Means groups observations around cluster centers. We test values of `k` from 2 to 10.

- **Inertia:** within-cluster sum of squared distances. Lower is better, but it always decreases as k increases.
- **Elbow method:** look for a point where the decrease in inertia begins to slow.
- **Silhouette score:** ranges from -1 to 1. Higher values indicate better separation and cohesion.

Use both measures together; do not choose k from the elbow curve alone.

In [ ]:
max_k = min(10, len(X_pca) - 1)
k_values = range(2, max_k + 1)
inertias = []
silhouette_scores = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_pca, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set_title("Elbow Method")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].grid(True)

axes[1].plot(list(k_values), silhouette_scores, marker="o")
axes[1].set_title("Silhouette Score by k")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")
axes[1].grid(True)
plt.tight_layout()
plt.show()

score_table = pd.DataFrame({
    "k": list(k_values),
    "inertia": inertias,
    "silhouette_score": silhouette_scores
})
display(score_table.round(4))

best_k_silhouette = int(score_table.loc[score_table["silhouette_score"].idxmax(), "k"])
print("Best k according to silhouette score:", best_k_silhouette)
print("Also inspect the elbow plot before making the final choice.")

## 6. Fit the selected K-Means model

The default selection below uses the highest silhouette score among k=2 to 10. Review the elbow curve as well. If the elbow suggests a different sensible k, change `chosen_k` manually and rerun this cell and later K-Means cells.

In [ ]:
chosen_k = best_k_silhouette  # Change this after inspecting the elbow plot if justified.

kmeans = KMeans(n_clusters=chosen_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_pca)

print("Chosen k:", chosen_k)
print("K-Means silhouette score:", round(silhouette_score(X_pca, kmeans_labels), 4))
print("Cluster counts:")
display(pd.Series(kmeans_labels, name="cluster").value_counts().sort_index())

## 7. Visualize K-Means clusters in 2D and 3D PCA space

The scatter plots show the first two or three principal components. These are projections of the data; they do not show every dimension used by the clustering algorithm.

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=kmeans_labels, cmap="viridis", alpha=0.75)
plt.title("K-Means Clusters – 2D PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

if X_scaled.shape[1] >= 3:
    fig = plt.figure(figsize=(9, 7))
    ax = fig.add_subplot(111, projection="3d")
    points = ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2],
                         c=kmeans_labels, cmap="viridis", alpha=0.75)
    ax.set_title("K-Means Clusters – 3D PCA Projection")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_zlabel("PC3")
    fig.colorbar(points, ax=ax, label="Cluster")
    plt.show()

## 8. DBSCAN clustering

DBSCAN groups nearby observations and marks isolated observations as noise (`-1`). It does not require a preselected number of clusters.

We use the PCA representation retaining 90% variance. The k-distance plot helps inspect a possible `eps` value. DBSCAN is sensitive to `eps` and `min_samples`, so try nearby values if it produces one cluster or labels almost every point as noise.

In [ ]:
min_samples = max(5, 2 * X_pca.shape[1])
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_pca)
distances, indices = neighbors_fit.kneighbors(X_pca)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 5))
plt.plot(k_distances)
plt.title(f"DBSCAN k-distance Plot (k={min_samples})")
plt.xlabel("Points sorted by distance")
plt.ylabel("Distance to k-th nearest neighbor")
plt.grid(True)
plt.show()

# Starting value only: inspect the k-distance curve and adjust if necessary.
eps_value = float(np.percentile(k_distances, 90))
print("Starting eps value:", round(eps_value, 4))

In [ ]:
dbscan = DBSCAN(eps=eps_value, min_samples=min_samples)
dbscan_labels = dbscan.fit_predict(X_pca)

unique_dbscan = set(dbscan_labels)
n_dbscan_clusters = len(unique_dbscan - {-1})
n_noise = int(np.sum(dbscan_labels == -1))

print("DBSCAN clusters (excluding noise):", n_dbscan_clusters)
print("Noise points:", n_noise)
print("Cluster/noise counts:")
display(pd.Series(dbscan_labels, name="DBSCAN label").value_counts().sort_index())

if n_dbscan_clusters >= 2:
    non_noise = dbscan_labels != -1
    dbscan_sil = silhouette_score(X_pca[non_noise], dbscan_labels[non_noise])
    print("DBSCAN silhouette (non-noise points only):", round(dbscan_sil, 4))
else:
    print("Silhouette score is not defined because DBSCAN found fewer than 2 clusters.")

## 9. Visualize DBSCAN clusters

Noise observations are shown as their own label (`-1`). If DBSCAN finds only one cluster or mostly noise, revisit `eps_value` using the k-distance plot.

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=dbscan_labels, cmap="tab10", alpha=0.75)
plt.title("DBSCAN Clusters – 2D PCA Projection (-1 = noise)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="DBSCAN label")
plt.grid(True)
plt.show()

## 10. Hierarchical (Agglomerative) clustering

Agglomerative clustering starts with individual observations and repeatedly merges the closest groups. Here we use Ward linkage, which merges groups to minimize the increase in within-cluster variance.

We use the same selected `chosen_k` as K-Means to make the cluster-count comparison easier. A dendrogram is shown for a sample of up to 100 observations because plotting every leaf can become crowded.

In [ ]:
hierarchical = AgglomerativeClustering(n_clusters=chosen_k, linkage="ward")
hier_labels = hierarchical.fit_predict(X_pca)

print("Hierarchical cluster counts:")
display(pd.Series(hier_labels, name="cluster").value_counts().sort_index())
print("Hierarchical silhouette score:", round(silhouette_score(X_pca, hier_labels), 4))

sample_n = min(100, len(X_pca))
sample_indices = np.random.default_rng(RANDOM_STATE).choice(len(X_pca), size=sample_n, replace=False)
Z = linkage(X_pca[sample_indices], method="ward")

plt.figure(figsize=(13, 5))
dendrogram(Z, no_labels=True)
plt.title(f"Hierarchical Clustering Dendrogram (sample of {sample_n} students)")
plt.xlabel("Sample observations")
plt.ylabel("Ward distance")
plt.show()

## 11. Visualize hierarchical clusters

In [ ]:
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=hier_labels, cmap="viridis", alpha=0.75)
plt.title("Hierarchical Clusters – 2D PCA Projection")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(scatter, label="Cluster")
plt.grid(True)
plt.show()

## 12. Compare cluster methods

Compare the silhouette scores for K-Means and hierarchical clustering. For DBSCAN, the score is calculated only for non-noise observations and only when at least two non-noise clusters exist. Silhouette scores across methods should be interpreted carefully because DBSCAN excludes noise points.

In [ ]:
comparison = [
    {"Method": "K-Means", "Clusters (excluding noise)": len(set(kmeans_labels)), 
     "Silhouette": silhouette_score(X_pca, kmeans_labels)},
    {"Method": "Hierarchical (Ward)", "Clusters (excluding noise)": len(set(hier_labels)),
     "Silhouette": silhouette_score(X_pca, hier_labels)}
]

if n_dbscan_clusters >= 2:
    comparison.append({
        "Method": "DBSCAN (non-noise only)",
        "Clusters (excluding noise)": n_dbscan_clusters,
        "Silhouette": silhouette_score(X_pca[dbscan_labels != -1], dbscan_labels[dbscan_labels != -1])
    })
else:
    comparison.append({
        "Method": "DBSCAN",
        "Clusters (excluding noise)": n_dbscan_clusters,
        "Silhouette": np.nan
    })

display(pd.DataFrame(comparison).round(4))

## 13. Interpret clusters using final mathematics grade

`G3` was not used as an input to PCA or clustering. We now attach it only to describe the resulting groups. These summaries are descriptive and do not show that cluster membership causes a grade difference.

In [ ]:
results = pd.DataFrame({
    "G3": grade.to_numpy(),
    "KMeans_cluster": kmeans_labels,
    "Hierarchical_cluster": hier_labels,
    "DBSCAN_cluster": dbscan_labels
}, index=grade.index)

print("K-Means cluster grade summary:")
display(results.groupby("KMeans_cluster")["G3"].agg(["count", "mean", "std", "median"]).round(2))

print("Hierarchical cluster grade summary:")
display(results.groupby("Hierarchical_cluster")["G3"].agg(["count", "mean", "std", "median"]).round(2))

print("DBSCAN cluster grade summary (including -1 noise group):")
display(results.groupby("DBSCAN_cluster")["G3"].agg(["count", "mean", "std", "median"]).round(2))

## 14. Findings – complete after running the notebook

Fill this section using the actual outputs generated above:

1. The dataset contained **[enter rows used]** student records and **[enter encoded feature count]** clustering features after preprocessing.
2. PCA retained **[enter component count]** components and explained **[enter cumulative percentage]** of the variance.
3. The elbow plot suggested **[enter k]** clusters. The highest silhouette score was obtained at **[enter k]** with a score of **[enter score]**.
4. K-Means identified **[describe cluster sizes]**.
5. DBSCAN identified **[enter number]** clusters and **[enter noise count]** noise observations using the selected parameters.
6. Hierarchical clustering identified **[describe cluster sizes]** groups.
7. The descriptive G3 summaries showed **[describe observed pattern, without claiming causation]**.

### Conclusion
PCA reduced the feature space while retaining most of the variation. K-Means, DBSCAN, and hierarchical clustering provided different ways to explore student profiles. The final interpretation should consider the cluster visualizations, evaluation scores, cluster sizes, parameter choices, and the limitations of the dataset. No method is guaranteed to reveal a single objectively correct grouping.

## 15. Limitations and reproducibility notes

- The dataset represents students from two Portuguese schools; findings may not generalize to other populations.
- `studytime` and several other variables are categorical/ordinal. One-hot encoding treats categories as separate indicators and does not encode their order.
- Standardization and Euclidean-distance methods can be sensitive to outliers and feature representation.
- PCA is a linear projection and may not preserve every nonlinear structure.
- DBSCAN results depend strongly on `eps` and `min_samples`.
- Hierarchical clustering is sensitive to the distance metric and linkage method.
- Cluster labels are arbitrary identifiers; cluster 0 is not inherently better than cluster 1.
- The final grade is used only for post-cluster description, not to form clusters.

## 16. Submission checklist

- [ ] Download the official UCI `student-mat.csv` dataset.
- [ ] Place it in `data/raw/student-mat.csv`.
- [ ] Install the packages in `requirements-task4.txt`.
- [ ] Open this notebook in Jupyter Notebook or Google Colab.
- [ ] Run all cells from top to bottom.
- [ ] Check that plots and output tables appear.
- [ ] Adjust DBSCAN `eps_value` if the clustering result is not meaningful; document the value used.
- [ ] Complete the Findings section using your actual results.
- [ ] Save the executed notebook with outputs.
- [ ] Upload the notebook to your public GitHub repository.
- [ ] Open the notebook on GitHub to verify it displays correctly.
- [ ] Submit the public GitHub notebook link in RabTech Academy Task 4.

**Expected proof:** A Jupyter Notebook (`.ipynb`) containing PCA scree plots, silhouette scores, and cluster visualizations.